# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

> All record sets, fields, and columns are referenced by their `@id` fields for unambiguous access.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
print(f"Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and field `@id`s.

In [ ]:
# List all record sets by @id and their fields
record_set_ids = [rs['@id'] for rs in dataset.metadata.jsonld.get('recordSet', [])]
print(f"Record sets (@id): {record_set_ids}\n")
for record_set in dataset.metadata.jsonld.get('recordSet', []):
    print(f"RecordSet @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - Field @id: {field['@id']}, Name: {field.get('name','')}, DataType: {field.get('dataType','')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @id values to extract data from
record_sets = [rs['@id'] for rs in dataset.metadata.jsonld.get('recordSet', [])]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {df.shape}")

# Show columns from the main (first) record set
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]
    print(f"Columns in DataFrame for record set @id = {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's run some basic EDA. We select a numeric field, filter records, normalize values, and, if available, group by categories.

In [ ]:
# Example: Select a numeric field from the first record set for analysis
# Please adjust the field_id as per actual field @id from section 2 output
# For demonstration, we use 'coverage' and 'group' as common mass spectrometry field names (replace as necessary)

main_record_set_id = record_sets[0] if len(record_sets) > 0 else None
df = dataframes[main_record_set_id]
# Try to find a likely numeric column
candidate_numeric_fields = [col for col in df.columns if any(w in col.lower() for w in ['coverage', 'peptide', 'mw', 'abundance', 'count', 'intensity', 'score', 'value', 'number', 'weight'])]
if len(candidate_numeric_fields) > 0:
    numeric_field = candidate_numeric_fields[0]  # Use the first match
else:
    # Fallback to first column
    numeric_field = df.columns[0]
print(f"Selected field for numeric analysis: {numeric_field}")

# Filter threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is None:
    print(f"The selected field {numeric_field} is not numeric!")
else:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try to find a likely grouping field
    candidate_group_fields = [col for col in df.columns if any(w in col.lower() for w in ['group', 'sample', 'type', 'class', 'category', 'condition'])]
    if len(candidate_group_fields) > 0:
        group_field = candidate_group_fields[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(grouped_df.head())

## 5. Visualization
Let's visualize distributions or relationships between the selected numeric field and (if present) a grouping variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group if available
if len(candidate_group_fields) > 0:
    group_field = candidate_group_fields[0]
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded and inspected dataset metadata via the Croissant schema
- Enumerated record sets and their field `@id`s
- Extracted and processed records using field `@id` references
- Performed simple EDA (filtering, normalization, grouping)
- Visualized numeric field distributions

Always refer to field and record set `@id` values (see section 2) for robust downstream referencing.